# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/7enno/HananAlawawdaRepository/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!git clone https://github.com/7enno/HananAlawawdaRepository.git
%cd HananAlawawdaRepository
import pandas as pd

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Select features
feature_cols = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_direction",
    "trend_pct"
]

X = df[feature_cols].copy()

# Fill missing numeric values with the median
numeric_cols = X.select_dtypes(include=["number"]).columns
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

# Fill missing categorical values
categorical_cols = X.select_dtypes(include=["object"]).columns
X[categorical_cols] = X[categorical_cols].fillna("Unknown")

# One-hot encode categorical features
X = pd.get_dummies(X, columns=categorical_cols)

print("Feature vector shape:", X.shape)
X.head()


Cloning into 'HananAlawawdaRepository'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 147 (delta 56), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 1.86 MiB | 10.39 MiB/s, done.
Resolving deltas: 100% (56/56), done.
/content/HananAlawawdaRepository/HananAlawawdaRepository/HananAlawawdaRepository/HananAlawawdaRepository/HananAlawawdaRepository
Feature vector shape: (30000, 30)


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,...,main_intent_Unknown,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,trend_direction_down,trend_direction_flat,trend_direction_new,trend_direction_stable,trend_direction_up
0,10.0,0.67,2.05,3221.0,20457.0,3803,29,0.76,10.6,5.88,...,False,False,False,False,True,True,False,False,False,False
1,90.0,0.01,0.05,2481.0,15562.0,15320,7,0.05,20.3,0.00,...,False,False,True,False,False,True,False,False,False,False
2,0.0,0.00,0.00,3515.0,23643.0,12581,11,0.09,36.5,0.00,...,False,False,True,False,False,True,False,False,False,False
3,10.0,0.00,0.00,2877.0,19116.0,11751,58,0.49,6.2,1.28,...,False,True,False,False,False,False,False,False,True,False
4,0.0,0.00,0.00,2803.0,17469.0,19140,24,0.13,44.0,0.00,...,False,False,True,False,False,True,False,False,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing values | Available before prediction? |
|---------|---------|----------------|------------------------------|
| search_volume | Estimated search demand for the topic | Fill with median | Yes |
| competition | Competition score | Fill with median | Yes |
| competition_level | Competition category (Low/Medium/High) | Fill with "Unknown" | Yes |
| cpc | Estimated cost per click | Fill with median | Yes |
| content_type | Type of content | Fill with "Unknown" | Yes |
| main_intent | Search intent | Fill with "Unknown" | Yes |
| word_count | Number of words | Fill with median | Yes |
| char_count | Number of characters | Fill with median | Yes |
| impressions_90d | Search impressions over the last 90 days | Fill with median | Yes |
| clicks_90d | Search clicks over the last 90 days | Fill with median | Yes |
| ctr | Click-through rate | Fill with median | Yes |
| avg_position | Average search position | Fill with median | Yes |
| engagement_rate | User engagement rate | Fill with median | Yes |
| scroll_rate | User scroll rate | Fill with median | Yes |
| ai_traffic_pct | Percentage of AI-generated traffic | Fill with median | Yes |
| trend_direction | Recent trend direction | Fill with "Unknown" | Yes |
| trend_pct | Percentage change in trend | Fill with median | Yes |

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*


I checked the selected features for possible data leakage.

- No identifier fields such as `content_id` or `client_id` are included in the feature vector.
- No target or label column is used as an input feature.
- The selected features are observable search and content metrics that are available before making the prediction.
- The dataset uses 90-day historical metrics, so the model relies on past observations rather than future information.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Identifier columns that should NOT be features
excluded = ["content_id", "client_id"]

print("Excluded columns:")
print(excluded)
print("\nAre excluded columns in the feature vector?")
for col in excluded:
    print(f"{col}: {col in X.columns}")
print("Feature vector shape:", X.shape)
print(X.columns.tolist()[:10])   # show first few feature names

Excluded columns:
['content_id', 'client_id']

Are excluded columns in the feature vector?
content_id: False
client_id: False
Feature vector shape: (30000, 30)
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'engagement_rate']


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*


| Field | Why excluded |
|--------|--------------|
| `content_id` | Unique identifier only. It does not describe page performance and could encourage the model to memorize specific pages. |
| `client_id` | Unique identifier for the client. It is not a meaningful predictive feature and could introduce bias or overfitting. |

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.